In [445]:
import numpy as np
import re
import chromadb
import ollama

### Loading a Pretrained Sentence Transformer Model

To generate text embeddings, we use the pretrained **all-MiniLM-L6-v2** model from the Sentence Transformers library.

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
```

#### Why this model?

* It is lightweight and efficient, making it suitable for experimentation and production use.
* It converts sentences into dense vector representations (**embeddings**) with a dimension of **384**.
* Sentences with similar meanings are mapped to nearby locations in the embedding space.
* It is commonly used for tasks such as:

  * Semantic search
  * Text similarity
  * Clustering
  * Information retrieval
  * Recommendation systems

After loading the model, each sentence can be transformed into a 384-dimensional embedding vector that captures its semantic meaning.


In [446]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10775.55it/s]


In [447]:
sentences = [
    "I am scared about the rest of my professional path.",
    "My career path is not clear at the moment, but with effort and persistence, I can move in the right direction.",
    "I like to know more about Germany.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
embeddings.shape

(3, 384)

# Use Built-in similarity Function

In [448]:
# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)

tensor([[1.0000, 0.5275, 0.0500],
        [0.5275, 1.0000, 0.0487],
        [0.0500, 0.0487, 1.0000]])


### Using NumPy Instead of the Built-in `similarity` Function

To check the similarity between two embeddings, we can implement **Cosine Similarity** ourselves using NumPy instead of relying on an internal similarity function.

Cosine Similarity measures how close two vectors are by calculating the cosine of the angle between them.

$$
\text{Cosine Similarity}(A,B)
=
\frac{A \cdot B}
{\|A\|\,\|B\|}
$$

The expanded form is:

$$
\cos(\theta)
=
\frac{\sum_{i=1}^{n} A_iB_i}
{\sqrt{\sum_{i=1}^{n} A_i^2}
 \sqrt{\sum_{i=1}^{n} B_i^2}}
$$

#### Key Points

* **Direction over magnitude:** Cosine Similarity focuses on the direction of the vectors rather than their magnitude or length.
* **Value range:** The result ranges from **-1 to 1**.
* **Interpretation:**

  * **1** → The vectors point in exactly the same direction → highly similar.
  * **0** → The vectors are perpendicular → no directional similarity.
  * **-1** → The vectors point in completely opposite directions.

This makes Cosine Similarity particularly useful for comparing **embeddings**, where the direction of the vector represents semantic information.


In [449]:
def cosine_sim(vec1, vec2):

    dot_product = np.dot(vec1 , vec2)
    norm = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    cosine_similarity = dot_product / norm
    return cosine_similarity


In [450]:
print(cosine_sim(embeddings[0], embeddings[1]))
print(cosine_sim(embeddings[0], embeddings[2]))
print(cosine_sim(embeddings[1], embeddings[2]))

# %%


0.5274734
0.050040323
0.048657864


In [451]:
ranking = {}

query = "Every body should make your own decision"
embeddings_query = model.encode(query)

for i, sentence in enumerate(sentences):
    score = cosine_sim(embeddings_query , embeddings[i])
    ranking[sentence] = score

ranking_sorted = sorted(
    ranking.items(),
    key=lambda x: x[1],
    reverse=True
)

In [452]:
print(ranking_sorted)

[('My career path is not clear at the moment, but with effort and persistence, I can move in the right direction.', np.float32(0.2622649)), ('I am scared about the rest of my professional path.', np.float32(0.212736)), ('I like to know more about Germany.', np.float32(0.053430203))]


In [453]:
for sentence, score in ranking_sorted:
    print(f"{score:.4f} | {sentence}")

0.2623 | My career path is not clear at the moment, but with effort and persistence, I can move in the right direction.
0.2127 | I am scared about the rest of my professional path.
0.0534 | I like to know more about Germany.


## Text Loading and Initial Chunking

In this step, the text of **Article 5 of the EU AI Act** was loaded from a local text file and inspected before applying any chunking strategy.

### 1. Load the text file

The article was loaded using Python's `open()` function with a context manager:

```python
with open('data/ai_act_article5.txt', 'r') as file:
    text = file.read()
```

Using a relative path makes the notebook more portable and allows it to work when the project is cloned to another machine, assuming the project structure remains the same.

### 2. Inspect the extracted text

The type and beginning of the text were checked:

```python
print(type(text))
print(text[:1000])
```

The text was confirmed to be stored as a Python `str`.

The total number of characters was also checked:

```python
print(len(text))
```

To inspect invisible characters such as newlines and spaces, `repr()` was used:

```python
print(repr(text[:300]))
```

The inspection showed that the document contains actual newline characters and that major sections are separated by `\n\n`.

### 3. Inspect newline structure

The number of newline characters was checked:

```python
print(text.count('\n'))
```

This confirmed that the text contains paragraph-style line breaks.

However, a simple split using:

```python
text.split('\n\n')
```

would not produce ideal semantic chunks because the lettered sections of Article 5 are structured like:

```text
(a)

the placing on the market...

(b)

...
```

A simple `\n\n` split would therefore separate `(a)` from its corresponding content.

### 4. Split using lettered section markers

Because Article 5 contains structured lettered sections such as `(a)`, `(b)`, and `(c)`, these markers were used to identify potential structural boundaries for the next stage of chunking.

Python's `re` module was used with `re.split()`:

```python
import re

parts = re.split(r'\n\n(\([a-z]\))\n\n', text)

print(len(parts))
```

The regular expression identifies single-letter parenthetical markers such as:

```text
(a)
(b)
(c)
(i)
```

The capturing group:

```text
(\([a-z]\))
```

keeps these markers in the resulting list.

The resulting structure alternates between text and matched lettered markers, allowing the original structure of the document to be inspected before deciding which markers should define the final chunk boundaries.

Some markers, such as `(i)`, are nested within larger sections rather than representing new top-level chunks. Therefore, the output of `re.split()` was inspected before recombining the parts into the final chunks.

### Current status

At this point:

* The Article 5 text has been loaded successfully.
* A relative file path is being used.
* The text structure has been inspected.
* Newline characters and formatting have been examined.
* A simple `\n\n` split was evaluated and found unsuitable for semantic chunking.
* A regex-based split was created to identify single-letter parenthetical markers.
* The lettered markers are preserved using a capturing group.
* The resulting parts were inspected to distinguish top-level sections from nested markers.

**Next step:** recombine the extracted parts into coherent chunks while preserving nested content within its corresponding top-level section.



In [454]:
with open('data/ai_act_article5.txt' , 'r') as file: 
    text = file.read()

In [455]:
print(f'Type of our text file is :{type(text)}')
print('--' * 30)
print(f'Length of our text file is :{len(text)}')
print('--' * 30)
print(text[:1000])


Type of our text file is :<class 'str'>
------------------------------------------------------------
Length of our text file is :11172
------------------------------------------------------------
Article 5

Prohibited AI practices

1.   The following AI practices shall be prohibited:

(a)

the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;

(b)

the placing on the market, the putting into service or the use of an AI system that exploits any of the vulnerabilities of 

In [456]:
text.count('\n')

84

In [457]:
repr(text[:300])

"'Article 5\\n\\nProhibited AI practices\\n\\n1.   The following AI practices shall be prohibited:\\n\\n(a)\\n\\nthe placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with '"

In [458]:
parts = re.split(r'\n\n(\([a-z]\))\n\n', text) 
print(len(parts))

25


In [459]:
print(parts[:5])

['Article 5\n\nProhibited AI practices\n\n1.   The following AI practices shall be prohibited:', '(a)', 'the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;', '(b)', 'the placing on the market, the putting into service or the use of an AI system that exploits any of the vulnerabilities of a natural person or a specific group of persons due to their age, disability or a specific social or economic situation, with the objective, or the effect, of materially distorting t

## 4. Recombining Labels and Contents into Final Chunks

After splitting the text with regular expressions, the extracted labels and their corresponding content were stored separately.

The labels were extracted using:

```python
labels = parts[1::2]
```

and the corresponding contents using:

```python
contents = parts[2::2]
```

However, not every single-letter marker represents a new top-level section. Some markers, such as `(i)`, are nested inside larger sections of Article 5.

Therefore, the extracted parts were recombined based on the expected top-level sections `(a)` through `(h)`.

```python
chunks = [parts[0]]

expected = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']

pointer = 0

for label, content in zip(labels, contents):

    if pointer < len(expected) and label[1:-1] == expected[pointer]:

        chunks.append(' '.join([label, content]))

        pointer += 1

    else:

        chunks[-1] += '\n' + label + ' ' + content
```

The `expected` list defines the top-level sections of Article 5 that should become separate chunks:

```text
(a) → chunk_1
(b) → chunk_2
(c) → chunk_3
(d) → chunk_4
(e) → chunk_5
(f) → chunk_6
(g) → chunk_7
(h) → chunk_8
```

When a nested marker such as `(i)` is encountered, it is not treated as a new top-level chunk. Instead, it is appended to the current chunk so that its relationship with the parent section is preserved.

The condition:

```python
pointer < len(expected)
```

also prevents the code from attempting to access an element outside the `expected` list after all top-level sections have been processed.

The final structure contains **9 chunks**:

* `chunks[0]` → Article 5 introduction
* `chunks[1]` → `(a)` + its content
* `chunks[2]` → `(b)` + its content
* `chunks[3]` → `(c)` + its nested content
* `chunks[4]` → `(d)` + its content
* `chunks[5]` → `(e)` + its content
* `chunks[6]` → `(f)` + its content
* `chunks[7]` → `(g)` + its content
* `chunks[8]` → `(h)` + its nested content and the remaining related provisions

The final number of chunks was verified using:

```python
print(len(chunks))
```

which returned:

```text
9
```

At this point, the Article 5 text has been transformed into structurally coherent chunks while preserving nested provisions within their corresponding top-level sections.

These chunks are now ready for the next stage: **generating embeddings and storing them for vector retrieval**.


In [460]:
chunks = [parts[0]]
labels = parts[1::2]
contents = parts[2::2]

print(labels)
print(contents[:3])

['(a)', '(b)', '(c)', '(i)', '(d)', '(e)', '(f)', '(g)', '(h)', '(i)', '(a)', '(b)']
['the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;', 'the placing on the market, the putting into service or the use of an AI system that exploits any of the vulnerabilities of a natural person or a specific group of persons due to their age, disability or a specific social or economic situation, with the objective, or the effect, of materially distorting the behaviour of that pers

In [461]:
expected = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
pointer = 0 

for label, content in zip(labels, contents):

    if pointer < len(expected) and label[1:-1] == expected[pointer]:

        chunks.append(' '.join([label , content]))
        pointer += 1
        
    else:
        chunks[-1] += '\n' + label + ' ' + content

In [462]:
len(chunks)

9

In [463]:
chunks[8]

'(h) the use of ‘real-time’ remote biometric identification systems in publicly accessible spaces for the purposes of law enforcement, unless and in so far as such use is strictly necessary for one of the following objectives:\n(i) the targeted search for specific victims of abduction, trafficking in human beings or sexual exploitation of human beings, as well as the search for missing persons;\n\n(ii)\n\nthe prevention of a specific, substantial and imminent threat to the life or physical safety of natural persons or a genuine and present or genuine and foreseeable threat of a terrorist attack;\n\n(iii)\n\nthe localisation or identification of a person suspected of having committed a criminal offence, for the purpose of conducting a criminal investigation or prosecution or executing a criminal penalty for offences referred to in Annex II and punishable in the Member State concerned by a custodial sentence or a detention order for a maximum period of at least four years.\n\nPoint (h) o

In [464]:
for i, chunk in enumerate(chunks):
    print(f"chunk_{i}: {repr(chunk[:100])}")

chunk_0: 'Article 5\n\nProhibited AI practices\n\n1.   The following AI practices shall be prohibited:'
chunk_1: '(a) the placing on the market, the putting into service or the use of an AI system that deploys subl'
chunk_2: '(b) the placing on the market, the putting into service or the use of an AI system that exploits any'
chunk_3: '(c) the placing on the market, the putting into service or the use of AI systems for the evaluation '
chunk_4: '(d) the placing on the market, the putting into service for this specific purpose, or the use of an '
chunk_5: '(e) the placing on the market, the putting into service for this specific purpose, or the use of AI '
chunk_6: '(f) the placing on the market, the putting into service for this specific purpose, or the use of AI '
chunk_7: '(g) the placing on the market, the putting into service for this specific purpose, or the use of bio'
chunk_8: '(h) the use of ‘real-time’ remote biometric identification systems in publicly accessible spaces for'


In [465]:
print(len(chunks))
print(chunks[1])

9
(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;


## Generate Chunk Embeddings

Generate an embedding vector for each Article 5 chunk using the pretrained `all-MiniLM-L6-v2` model.

Each chunk is represented as a 384-dimensional vector, allowing us to compare the semantic similarity between chunks and queries.

In [466]:
chunks_embedding = model.encode(chunks)

print(chunks_embedding.shape)
print(len(chunks))


(9, 384)
9


## Vector Storage & Retrieval

## Vector Database Setup

Create a persistent Chroma client to store embeddings and documents locally.

The database is stored in the `vector_store/` directory so that the collection persists between notebook sessions.

In [467]:
client = chromadb.PersistentClient(path='vector_store')

## Create a Chroma Collection

Create a collection for the EU AI Act Article 5 chunks.

A collection is a named space in Chroma where related documents and their embeddings are stored.

In [468]:
collection = client.get_or_create_collection(name='ai_act_article5')

In [469]:
cosine_collection = client.get_or_create_collection(
    name="ai_act_article5_v2",
    metadata={"hnsw:space": "cosine"}
)

## Vector Collection Versions

During development, two Chroma collections were created to keep the original implementation separate from the corrected version.

### Original Collection

```python
collection = client.get_or_create_collection(
    name="ai_act_article5"
)
```

This was the original collection created during the first implementation.

It contains the embeddings generated from the initial **13-chunk structure**, which was later found to have an issue in the chunking logic. This collection is kept as a record of the initial implementation but is no longer used for the main retrieval pipeline.

### Corrected Collection

```python
cosine_collection = client.get_or_create_collection(
    name="ai_act_article5_v2",
    metadata={"hnsw:space": "cosine"}
)
```

A new collection was created after correcting the chunking strategy.

The corrected version contains **9 chunks** and uses **cosine distance** for vector retrieval.

Keeping the corrected collection separate prevents the old and new embeddings from being mixed and makes it possible to evaluate the corrected implementation independently.

### Current Retrieval Pipeline

From this point onward, the retrieval pipeline uses:

```text
ai_act_article5_v2
        ↓
9 corrected chunks
        ↓
384-dimensional embeddings
        ↓
Cosine distance
        ↓
Retrieval
        ↓
Generation
        ↓
Answer verification
```

The original `ai_act_article5` collection is retained only as a record of the earlier implementation.


## Store Corrected Chunks and Embeddings

Store the corrected Article 5 chunks and their corresponding embeddings in the new Chroma collection.

Each chunk receives a unique ID so that it can be identified during retrieval.

The corrected chunks are stored in the `ai_act_article5_v2` collection, which uses cosine distance for vector retrieval.

```python
cosine_collection.add(
    ids=[f'chunk_{i}' for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunks_embedding
)
```

## Verify Corrected Collection

After storing the chunks, the number of documents in the collection is checked:

```python
print(cosine_collection.count())
```

The collection should contain **9 chunks**:

* `chunk_0` → Article 5 introduction
* `chunk_1` → `(a)`
* `chunk_2` → `(b)`
* `chunk_3` → `(c)` and its nested content
* `chunk_4` → `(d)`
* `chunk_5` → `(e)`
* `chunk_6` → `(f)`
* `chunk_7` → `(g)`
* `chunk_8` → `(h)` and its related nested and subsequent provisions

The expected count is:

```text
9
```


### Original Vector Collection — Previous Version

The following code was used to add the initially generated chunks and embeddings to the original Chroma collection:

```python
collection.add(
    ids=[f'chunk_{i}' for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunks_embedding
)

print(collection.count())
```

This collection belongs to the **initial implementation**, which used the original 13-chunk structure.

After inspecting and correcting the chunking logic, a new collection was created using the corrected 9-chunk structure and cosine distance.

Therefore, this code is kept as a record of the original implementation but is **not used in the current retrieval pipeline**.

The current pipeline uses the corrected `cosine_collection` instead.


In [470]:
cosine_collection.add(
    ids=[f'chunk_{i}' for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunks_embedding
)

print(cosine_collection.count())

9


## Test Retrieval with a Query

Test semantic retrieval using a natural-language question:

> "What counts as manipulating someone's behavior?"

The query is converted into an embedding and compared against the stored Article 5 chunk embeddings.

## Generate Query Embedding

Convert the user's query into the same embedding space as the stored document chunks.

Using the same embedding model allows Chroma to compare the query vector with the chunk vectors.

## Retrieve Relevant Chunks

Query the Chroma collection and retrieve the three chunks with the smallest distances to the query embedding.

Lower distance indicates greater similarity between the query and the retrieved chunk.

In [471]:
query = "What counts as manipulating someone's behavior?"

query_embedding = model.encode(query)

In [472]:
results = cosine_collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

## Inspect Retrieval Results

Inspect the retrieved chunk IDs, documents, and distances to understand which parts of Article 5 were considered most relevant to the query.

This allows us to evaluate whether the retrieval result matches our expectations.

In [473]:
print(results)

{'ids': [['chunk_1', 'chunk_3', 'chunk_2']], 'embeddings': None, 'documents': [['(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;', '(c) the placing on the market, the putting into service or the use of AI systems for the evaluation or classification of natural persons or groups of persons over a certain period of time based on their social behaviour or known, inferred or predicted personal or personality characteristics, with the social score leading to either

## Compare Distance Metrics

A second Chroma collection was created using cosine distance instead of the default L2 (Euclidean) distance.

This allowed the effect of the distance metric on retrieval ranking to be investigated.


## Store Embeddings in the Cosine Collection

The same Article 5 chunks and embeddings were stored in the cosine-distance collection.

Using the same chunks and embeddings allows the two distance metrics to be compared fairly.


## Test Retrieval with Cosine Distance

The same query was run against the cosine-distance collection.

The retrieved chunks and their distances were compared with the results from the L2 collection.

## L2 vs. Cosine Retrieval

For this initial experiment, both L2 and cosine distance returned the same retrieval ranking:

chunk_4
chunk_1
chunk_2

Although the distance values were different, the ranking remained unchanged.

This indicated that the distance metric was not responsible for the unexpected retrieval result observed during this experiment.

Note: This experiment was performed before a chunking issue was discovered and corrected. The chunk IDs above refer to the original chunking structure.

### The original collections (ai_act_article5 and ai_act_article5_cosine) were created before a chunking bug was discovered. A corrected collection (ai_act_article5_v2) was later created using the revised chunking logic and is used for all subsequent retrieval and evaluation experiments.

In [474]:
print(cosine_collection.count())

9


In [475]:
print(cosine_collection.name)

ai_act_article5_v2


## Query Phrasing Experiment

The first query produced an unexpected top result. To investigate whether retrieval is sensitive to query wording, test the same collection with a more precise query.

In [476]:
query_2 = "deceptive AI techniques that distort behavior"
query_2_embedding = model.encode(query_2)

In [477]:
results_2_cosine = cosine_collection.query(
    query_embeddings=[query_2_embedding],
    n_results=3
)

print(results_2_cosine["ids"])
print(results_2_cosine["distances"])

[['chunk_1', 'chunk_4', 'chunk_0']]
[[0.5411045551300049, 0.6588382720947266, 0.6689181327819824]]


## Conclusion

Built a complete semantic retrieval pipeline using embeddings and ChromaDB to search Article 5 of the EU AI Act.

I tested both L2 and cosine distance and found that they produced the same retrieval ranking. The experiments also showed that retrieval quality depends strongly on query phrasing: a more precise query was able to retrieve the most relevant chunk correctly.

In [478]:
def retrieve(query , n = 3):

    query_embedding = model.encode(query)
    results = cosine_collection.query(
            query_embeddings= [query_embedding],
            n_results=n
         )
    documents = results['documents'][0]
    distances = results['distances'][0]
    ids = results['ids'][0]

    return list(zip(ids, documents, distances))


In [479]:
retrieve("deceptive AI techniques that distort behavior")

[('chunk_1',
  '(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;',
  0.5411045551300049),
 ('chunk_4',
  '(d) the placing on the market, the putting into service for this specific purpose, or the use of an AI system for making risk assessments of natural persons in order to assess or predict the risk of a natural person committing a criminal offence, based solely on the profiling of a natural person or on assessing their personality traits and characteristics; 

In [480]:
response = ollama.chat(
    model="mistral",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of Germany?"
        }
    ]
)

In [481]:
print(response["message"]["content"])

 The capital of Germany is Berlin. Berlin is a significant city, rich in history and culture, known for its vibrant arts scene, iconic landmarks like the Berlin Wall, and the Brandenburg Gate. It has been the capital of Germany since German reunification in 1990.


In [482]:
def generate_answer(query, n=3):
    results = retrieve(query, n)

    documents = [document for chunk_id, document, distance in results]

    context = "\n\n".join(documents)

    prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say I don't know.

Context:
{context}

Question:
{query}
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"] , context         

In [483]:
answer, context = generate_answer("What counts as manipulating someone's behavior?")
print(f'Answer is : {answer}')
print('-'*100)
print(f'Context is : {context}')

Answer is :  Using an AI system that deploys subliminal techniques beyond a person's consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behavior of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken, counts as manipulating someone's behavior. (Based on context (a))

Additionally, the use of AI systems for the evaluation or classification of natural persons or groups of persons over a certain period of time based on their social behavior or known, inferred or predicted personal or personality characteristics, which leads to detrimental or unfavourable treatment of certain natural persons or groups of persons in social contexts unrelated to the contexts in which the data was originally generated or collected, or detrimental or unfavourable treatment that is unjustified or dispropor

In [484]:
answer, context = generate_answer("What is the capital of Germany?")
print(f'Answer is : {answer}')
print('-'*100)
print(f'Context is : {context}')

Answer is :  I don't know. The provided context does not include information about the capital of Germany.
----------------------------------------------------------------------------------------------------
Context is : (f) the placing on the market, the putting into service for this specific purpose, or the use of AI systems to infer emotions of a natural person in the areas of workplace and education institutions, except where the use of the AI system is intended to be put in place or into the market for medical or safety reasons;

(d) the placing on the market, the putting into service for this specific purpose, or the use of an AI system for making risk assessments of natural persons in order to assess or predict the risk of a natural person committing a criminal offence, based solely on the profiling of a natural person or on assessing their personality traits and characteristics; this prohibition shall not apply to AI systems used to support the human assessment of the involveme

In [485]:
answer , context =  generate_answer("What penalties apply if a company violates Article 5?")
print(f'Answer is : {answer}')
print('-'*100)
print(f'Context is : {context}')

Answer is :  I don't know, as the provided context does not specify any penalties for violating Article 5.
----------------------------------------------------------------------------------------------------
Context is : Article 5

Prohibited AI practices

1.   The following AI practices shall be prohibited:

(b) the placing on the market, the putting into service or the use of an AI system that exploits any of the vulnerabilities of a natural person or a specific group of persons due to their age, disability or a specific social or economic situation, with the objective, or the effect, of materially distorting the behaviour of that person or a person belonging to that group in a manner that causes or is reasonably likely to cause that person or another person significant harm;

(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with t

# Generation: From Retrieved Context to an Answer

At this stage, the retrieval system can find the most relevant chunks from the EU AI Act Article 5.

However, retrieval alone does not answer the user's question. Retrieval only finds evidence.

To produce a natural-language response, the retrieved context must be passed to a generative model.

The pipeline is:

```text
User Question
      ↓
Retrieval
      ↓
Relevant Chunks
      ↓
Context
      ↓
Prompt
      ↓
Mistral
      ↓
Generated Answer
```

For generation, I use **Mistral locally through Ollama**.

---

# The `generate_answer()` Function

The main function connects retrieval and generation:

```python
def generate_answer(query, n=3):

    results = retrieve(query, n)

    documents = [
        document
        for chunk_id, document, distance in results
    ]

    context = "\n\n".join(documents)

    prompt = f"""
Answer the question using only the context below.

If the answer is not in the context, say I don't know.

Context:

{context}

Question:

{query}
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]

    return answer, context
```

---

# What Does This Function Do?

The function combines the two main components of a basic RAG system:

* **Retrieval** → finds relevant information.
* **Generation** → uses that information to produce an answer.

The retrieval results now contain three pieces of information:

```text
(chunk_id, document, distance)
```

For example:

```text
[
    ("chunk_0", "Article 5 ...", 0.20),
    ("chunk_7", "(f) ...", 0.54),
    ("chunk_1", "(a) ...", 0.55)
]
```

The `chunk_id` identifies the source chunk, the `document` contains its text, and the `distance` indicates how close the chunk is to the query in embedding space.

---

# 1. Retrieve Relevant Chunks

```python
results = retrieve(query, n)
```

The `retrieve()` function searches the ChromaDB collection and returns the most relevant chunks.

Each result contains:

```text
(chunk_id, document, distance)
```

For example:

```text
[
    ("chunk_0", "chunk text...", 0.54),
    ("chunk_2", "another chunk...", 0.65),
    ("chunk_1", "another chunk...", 0.67)
]
```

The distance value is useful for retrieval analysis, while the chunk ID allows the evaluation system to identify which source chunks were retrieved.

---

# 2. Extract the Documents

```python
documents = [
    document
    for chunk_id, document, distance in results
]
```

For generation, only the chunk text is needed.

The `chunk_id` and `distance` are not passed to the language model as part of the context.

They remain available in the retrieval results for evaluation and analysis.

---

# 3. Build the Context

```python
context = "\n\n".join(documents)
```

The retrieved chunks are combined into a single context string.

The double newline visually separates chunks:

```text
Chunk 1

Chunk 2

Chunk 3
```

This helps the model distinguish between different pieces of evidence.

The resulting context becomes the source information provided to Mistral.

---

# 4. Build the RAG Prompt

The prompt contains three components:

## Instruction

```text
Answer the question using only the context below.

If the answer is not in the context, say I don't know.
```

This instruction encourages grounded generation and reduces hallucination.

## Context

```text
Context:

{context}
```

The retrieved evidence from Article 5.

## Question

```text
Question:

{query}
```

The user's original question.

The final prompt structure is:

```text
Instruction
     +
Context
     +
Question
```

---

# 5. Generate the Answer with Mistral

```python
response = ollama.chat(
    model="mistral",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
```

The prompt is sent to the locally running Mistral model through Ollama.

Mistral reads the retrieved context and generates a response based on that information.

---

# 6. Return Both the Answer and Context

```python
answer = response["message"]["content"]

return answer, context
```

The function returns two values:

1. The generated answer.
2. The exact context used during generation.

Example:

```python
answer, context = generate_answer(
    "What counts as manipulating someone's behavior?"
)
```

Returning the context is important because the next stage of the project will verify whether the generated answer is actually supported by the evidence that produced it.

This means generation and verification can use the **same retrieved context**, rather than performing retrieval again.

---

# Grounded Generation

A key goal of this stage is grounding.

Grounded generation means the model should answer using the retrieved source material rather than relying on its own general knowledge.

The instruction:

```text
Answer the question using only the context below.
```

helps reduce hallucination by constraining the model to the provided evidence.

However, prompting alone is **not a guarantee of correctness**.

A language model can still generate unsupported information even when explicitly instructed to use only the provided context.

This is why the next stage introduces a separate verification step.

---

# Testing the RAG System

The system was tested using both answerable and unanswerable questions.

## Test 1: Question Outside the Context

```python
answer, context = generate_answer(
    "What is the capital of Germany?"
)
```

The retrieved context still comes from Article 5.

Since the answer is not present in the retrieved evidence, the model responded:

```text
I don't know. The context does not contain information about the capital of Germany.
```

This demonstrates that the model followed the grounding instruction instead of answering from its general knowledge.

---

## Test 2: Related Topic but Missing Information

```python
answer, context = generate_answer(
    "What penalties apply if a company violates Article 5?"
)
```

This is a stronger grounding test.

The question is related to Article 5, so retrieval can return semantically relevant chunks.

However, Article 5 itself does not provide the requested penalties.

The model responded:

```text
I don't know. The context does not specify any penalties for violating Article 5.
```

This indicates that the model recognized the absence of supporting evidence rather than inventing an answer.

---

# Why Test 2 Matters

There is an important difference between these two scenarios.

## Scenario 1

```text
Question: What is the capital of Germany?

Context: EU AI Act Article 5
```

The question and context are clearly unrelated.

## Scenario 2

```text
Question: What penalties apply if a company violates Article 5?

Context: EU AI Act Article 5
```

The question is highly related to the retrieved context.

However, the specific answer is still missing.

This is a stronger grounding test because the model must recognize that relevant context exists while also recognizing that the requested information does not.

---

# Current Architecture

At this stage, the system architecture is:

```text
                    ┌───────────────┐
                    │ User Question │
                    └───────┬───────┘
                            ↓
                    ┌───────────────┐
                    │   Retrieval   │
                    │   ChromaDB    │
                    └───────┬───────┘
                            ↓
                ┌───────────────────────┐
                │ Retrieved Results     │
                │                       │
                │ chunk_id              │
                │ document              │
                │ distance              │
                └───────────┬───────────┘
                            ↓
                    ┌───────────────┐
                    │    Context    │
                    └───────┬───────┘
                            ↓
                    ┌───────────────┐
                    │   Generation  │
                    │    Mistral    │
                    └───────┬───────┘
                            ↓
              ┌───────────────────────────┐
              │ Answer + Context Used    │
              └────────────┬──────────────┘
                           ↓
                     Verification
                        (Next)
```

---

# Key Takeaways

* Retrieval finds relevant information.
* Each retrieved result contains a **chunk ID, document, and distance*


## Verification Agent

After generating an answer from the retrieved context, the system needs to check whether the answer is actually supported by that context.

For this purpose, I added a **verification agent** using an LLM as a judge.

The verifier receives:

* the retrieved context
* the generated answer

and evaluates whether the claims in the answer are supported by the provided context.

The verifier returns one of two decisions:

* `GROUNDED` — the answer is supported by the context.
* `NOT_GROUNDED` — the answer contains information that is not supported by the context.

This creates a second evaluation step after generation:

```text
Question
   ↓
Retrieval
   ↓
Retrieved Context
   ↓
Generation
   ↓
Generated Answer
   ↓
Verification
   ↓
GROUNDED / NOT_GROUNDED
```

The purpose of this step is to detect **hallucinated or unsupported claims**, even when the generated answer sounds plausible.

The verifier is intentionally separated from the generation step so that the answer is evaluated against the retrieved evidence rather than being trusted automatically.


In [486]:
def verify_answer(answer, context):
    prompt = f"""You are checking whether an answer is fully supported by a given context.
            If the answer correctly states that the context does not contain enough information to answer the question,
            treat this as GROUNDED, since accurately reporting an absence of information is itself a supported claim.

Context:
{context}

Answer:
{answer}

Is every claim in the answer supported by the context?
Respond with exactly one word on the first line — GROUNDED or NOT_GROUNDED — 
followed by a brief explanation on the next line."""

    response = ollama.chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    result = response["message"]["content"]

    lines = result.split('\n')
    clean_lines = []
    for line in lines:
        if line.strip():
            clean_lines.append(line.strip())

    first_line = clean_lines[0]

    if first_line.upper().startswith("NOT_GROUNDED"):
        verdict = "NOT_GROUNDED"
        remainder = first_line[len("NOT_GROUNDED"):]
    elif first_line.upper().startswith("GROUNDED"):
        verdict = "GROUNDED"
        remainder = first_line[len("GROUNDED"):]
    else:
        verdict = first_line.upper()
        remainder = ""

    explanation = (remainder.strip(" .") + " " + "\n".join(clean_lines[1:])).strip()

    return verdict, explanation
    

In [487]:
verdict, explanation = verify_answer(answer, context)

print("Verdict:", verdict)
print("Explanation:", explanation)

Verdict: GROUNDED
Explanation: The answer correctly reports that the provided context does not contain enough information to answer the question about penalties for violating Article 5.


In [488]:
query = "What counts as manipulating someone's behavior?"

answer, context = generate_answer(query)

print("Answer:")
print(answer)

print("-" * 100)

verdict, explanation = verify_answer(answer, context)

print("Verdict:", verdict)

print("Explanation:", explanation)

Answer:
 According to the context provided, manipulating someone's behavior involves the use of AI systems that deploy subliminal techniques or deceptive techniques with the objective or effect of materially distorting a person's ability to make an informed decision, causing them to take a decision they would not have otherwise taken, and causing significant harm. This can also include AI systems used for the evaluation or classification of individuals or groups over a certain period, leading to detrimental or unfavourable treatment unjustified or disproportionate to their social behavior or its gravity. Lastly, it includes AI systems that exploit vulnerabilities of a person or a specific group due to their age, disability, or specific social or economic situation, with the objective or effect of causing significant harm.
----------------------------------------------------------------------------------------------------
Verdict: GROUNDED
Explanation: Every claim in the answer is direc

## Testing the Verification Agent

Before running the complete evaluation, I tested the verification agent with two controlled cases.

### Test 1 — Grounded Answer

The first test used an answer that was directly supported by the retrieved Article 5 context.

The expected result was:

```text
GROUNDED
```

This test checks whether the verifier can correctly recognize an answer that is supported by the provided evidence.

### Test 2 — Fabricated Information

The second test intentionally introduced a fabricated claim about a **€35 million fine** that was not present in the retrieved context.

The expected result was:

```text
NOT_GROUNDED
```

This test was designed to simulate a hallucination and verify that the verification agent can detect unsupported information.

These two tests provide complementary checks:

* a supported answer should be accepted;
* an unsupported fabricated claim should be rejected.

This gave me a basic sanity check of the verifier before using it on the full evaluation set.


In [489]:
fake_answer = "Manipulating someone's behavior is punishable by a fine of up to €35 million under Article 5."

verdict, explanation = verify_answer(fake_answer, context)

print(verdict)
print(explanation)

NOT_GROUNDED
The context does not provide information about any potential penalties for manipulating someone's behavior, therefore the claim about a fine under Article 5 is not supported by the provided context.


In [490]:
for i , chunk in enumerate(chunks):
    print(f'Index chunk is : {i}')
    print()
    print(chunk)
    print("-" * 80)

Index chunk is : 0

Article 5

Prohibited AI practices

1.   The following AI practices shall be prohibited:
--------------------------------------------------------------------------------
Index chunk is : 1

(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm;
--------------------------------------------------------------------------------
Index chunk is : 2

(b) the placing on the market, the putting into service or the use of an AI system that exploits any of t

In [491]:
evaluation_questions = [
    {
        "question": "What kinds of AI practices are prohibited?",
        "expected_chunk_id": 0,
        "answerable": True
    },
    {
        "question": "What kind of AI manipulation can cause significant harm to people?",
        "expected_chunk_id": 1,
        "answerable": True
    },
    {
        "question": "How can AI take advantage of a person's vulnerabilities and cause harm?",
        "expected_chunk_id": 2,
        "answerable": True
    },
    {
        "question": "How can AI systems use people's social behavior to evaluate or score them?",
        "expected_chunk_id": 3,
        "answerable": True
    },
    {
        "question": "How can a social score lead to unfair treatment of people?",
        "expected_chunk_id": 3,
        "answerable": True
    },
    {
        "question": "Can AI be used to predict whether someone will commit a crime based only on their personality?",
        "expected_chunk_id": 4,
        "answerable": True
    },
    {
        "question": "Can AI systems collect people's faces from the internet to build facial recognition databases?",
        "expected_chunk_id": 5,
        "answerable": True
    },
    {
        "question": "Can AI be used to detect people's emotions at work or in education?",
        "expected_chunk_id": 6,
        "answerable": True
    },
    {
        "question": "Can AI infer sensitive personal characteristics from someone's biometric data?",
        "expected_chunk_id": 7,
        "answerable": True
    },
    {
        "question": "When can law enforcement use real-time biometric identification in public places?",
        "expected_chunk_id": 8,
        "answerable": True
    },
    {
        "question": "What situations can justify the use of real-time biometric identification by law enforcement?",
        "expected_chunk_id": 8,
        "answerable": True
    },
    {
        "question": "What factors should be considered before using real-time biometric identification in a public place?",
        "expected_chunk_id": 8,
        "answerable": True
    },
    {
        "question": "What should authorities consider about the potential harm before using a biometric identification system?",
        "expected_chunk_id": 8,
        "answerable": True
    },
    {
        "question": "What approval is needed before law enforcement can use real-time biometric identification in public places?",
        "expected_chunk_id": 8,
        "answerable": True
    },
    {
        "question": "What happens if the use of a real-time biometric identification system is not authorised?",
        "expected_chunk_id": 8,
        "answerable": True
    }
]


In [492]:
unanswerable_questions = [
    {
        "question": "What is the capital of Germany?",
        "expected_chunk_id": None,
        "answerable": False
    },
    {
        "question": "What penalties can companies face for using prohibited AI systems?",
        "expected_chunk_id": None,
        "answerable": False
    },
    {
        "question": "What fines can companies receive for violating AI regulations?",
        "expected_chunk_id": None,
        "answerable": False
    },
    {
        "question": "What rights do individuals have when an AI system causes them harm?",
        "expected_chunk_id": None,
        "answerable": False
    },
    {
        "question": "Who is responsible for monitoring companies that use AI systems?",
        "expected_chunk_id": None,
        "answerable": False
    }
]

eval_set = evaluation_questions + unanswerable_questions

In [493]:
len(eval_set)

20

## Corrected Evaluation Mapping After Chunking Fix

The evaluation set was originally created using the initial chunk structure.

After discovering and fixing the chunking problem, the Article 5 document was reorganized into **9 corrected chunks** instead of the original 13 chunks.

Because the chunk IDs changed, the expected chunk IDs in the evaluation set also had to be updated.

The corrected mapping reflects the new chunk structure.

Several questions are mapped to `chunk_8` because the corresponding Article 5 content about **real-time remote biometric identification** is contained within the same corrected chunk.

Therefore, the repeated appearance of `chunk_8` is intentional and does not indicate duplicated evaluation data.

The evaluation set was updated only after verifying the content of the corrected chunks, so each expected chunk ID corresponds to the location where the relevant information is now stored.


In [494]:
eval_set

[{'question': 'What kinds of AI practices are prohibited?',
  'expected_chunk_id': 0,
  'answerable': True},
 {'question': 'What kind of AI manipulation can cause significant harm to people?',
  'expected_chunk_id': 1,
  'answerable': True},
 {'question': "How can AI take advantage of a person's vulnerabilities and cause harm?",
  'expected_chunk_id': 2,
  'answerable': True},
 {'question': "How can AI systems use people's social behavior to evaluate or score them?",
  'expected_chunk_id': 3,
  'answerable': True},
 {'question': 'How can a social score lead to unfair treatment of people?',
  'expected_chunk_id': 3,
  'answerable': True},
 {'question': 'Can AI be used to predict whether someone will commit a crime based only on their personality?',
  'expected_chunk_id': 4,
  'answerable': True},
 {'question': "Can AI systems collect people's faces from the internet to build facial recognition databases?",
  'expected_chunk_id': 5,
  'answerable': True},
 {'question': "Can AI be used to

In [495]:
case = eval_set[0]
print(case)

{'question': 'What kinds of AI practices are prohibited?', 'expected_chunk_id': 0, 'answerable': True}


In [496]:
question = case['question']
expected_chunk_id = case['expected_chunk_id']
answerable = case['answerable']


In [497]:
print(question)
print(expected_chunk_id)
print(answerable)

What kinds of AI practices are prohibited?
0
True


In [498]:
retrieved = retrieve(question)
print(retrieved)

[('chunk_0', 'Article 5\n\nProhibited AI practices\n\n1.   The following AI practices shall be prohibited:', 0.20442050695419312), ('chunk_6', '(f) the placing on the market, the putting into service for this specific purpose, or the use of AI systems to infer emotions of a natural person in the areas of workplace and education institutions, except where the use of the AI system is intended to be put in place or into the market for medical or safety reasons;', 0.5437382459640503), ('chunk_1', '(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably

In [499]:
for case in eval_set:
    print(case["question"])

What kinds of AI practices are prohibited?
What kind of AI manipulation can cause significant harm to people?
How can AI take advantage of a person's vulnerabilities and cause harm?
How can AI systems use people's social behavior to evaluate or score them?
How can a social score lead to unfair treatment of people?
Can AI be used to predict whether someone will commit a crime based only on their personality?
Can AI systems collect people's faces from the internet to build facial recognition databases?
Can AI be used to detect people's emotions at work or in education?
Can AI infer sensitive personal characteristics from someone's biometric data?
When can law enforcement use real-time biometric identification in public places?
What situations can justify the use of real-time biometric identification by law enforcement?
What factors should be considered before using real-time biometric identification in a public place?
What should authorities consider about the potential harm before using

In [500]:
retrieval_results = []

for case in eval_set:

    question = case['question']
    retrieved = retrieve(question)
    chunk_ids = [chunk_id for chunk_id, document, distance in retrieved]
    
    if case["answerable"]:

        expected_chunk_id = case["expected_chunk_id"]
        expected_chunk = f"chunk_{expected_chunk_id}"
        retrieval_success = expected_chunk in chunk_ids

    else:

        expected_chunk = None
        retrieval_success = None

    retrieval_results.append({
        "question": question,
        "expected_chunk": expected_chunk,
        "retrieved_chunks": chunk_ids,
        "retrieval_success": retrieval_success,
        "answerable": case["answerable"],
    })

In [501]:
len(retrieval_results)

20

In [502]:
print(retrieval_results[0])

{'question': 'What kinds of AI practices are prohibited?', 'expected_chunk': 'chunk_0', 'retrieved_chunks': ['chunk_0', 'chunk_6', 'chunk_1'], 'retrieval_success': True, 'answerable': True}


In [503]:
generation_results = []

for case in eval_set:

    question = case["question"]
    answer, context = generate_answer(question)
    verdict, explanation = verify_answer(answer, context)
    generation_results.append({
        "question": question,
        "answerable": case["answerable"],
        "answer": answer,
        "verdict": verdict,
        "explanation": explanation
    })

In [504]:
len(generation_results)

20

# Evaluation Dataset Construction

At this stage, an evaluation dataset was created to systematically test the retrieval and generation performance of the RAG system.

The dataset contains both answerable and unanswerable questions derived from Article 5 of the EU AI Act.

The goal is to evaluate:

* Retrieval accuracy
* Refusal behavior on unsupported questions
* Groundedness of generated answers

The evaluation set includes:

* 15 answerable questions linked to specific Article 5 chunks
* 5 unanswerable questions whose answers are not contained in the provided context

Each answerable question is associated with an expected chunk ID, allowing retrieval performance to be measured automatically.

Example structure for answerable questions:

```python
{
    "question": "What kinds of AI practices are prohibited?",
    "expected_chunk_id": 0,
    "answerable": True
}
```

Example structure for unanswerable questions:

```python
{
    "question": "What is the capital of Germany?",
    "expected_chunk_id": None,
    "answerable": False
}
```

The answerable questions were designed to resemble realistic user queries rather than direct restatements of the legal text. This helps evaluate whether the retrieval system can match natural-language questions to the correct regulatory content.

The unanswerable questions serve a different purpose. They test whether the system can recognize when the required information is not present in the retrieved context and avoid generating unsupported answers.

This evaluation dataset will be used in the next stage to automatically assess:

1. Retrieval performance
2. Refusal behavior on unsupported queries
3. Groundedness of generated responses

By combining answerable and unanswerable cases, the evaluation process can measure not only retrieval quality, but also hallucination resistance and the ability of the system to remain grounded in the provided evidence.


## Filter Answerable Cases

Before calculating retrieval accuracy, only answerable questions are selected from the evaluation results.

Unanswerable questions are excluded because they do not have a correct target chunk and therefore cannot be used to measure retrieval performance.

This filtered subset will be used to calculate retrieval accuracy.

In [505]:
answerable_retrieval = [
    r for r in retrieval_results
    if r["answerable"]
]

len(answerable_retrieval)

15

In [506]:
print(answerable_retrieval[0])

{'question': 'What kinds of AI practices are prohibited?', 'expected_chunk': 'chunk_0', 'retrieved_chunks': ['chunk_0', 'chunk_6', 'chunk_1'], 'retrieval_success': True, 'answerable': True}


In [507]:
successful_retrievals = sum(
    r["retrieval_success"] == True
    for r in answerable_retrieval
)

print(successful_retrievals)

15


In [508]:
retrieval_accuracy = (
    successful_retrievals / len(answerable_retrieval)
) * 100

print(f"Retrieval Accuracy: {retrieval_accuracy:.2f}%")

Retrieval Accuracy: 100.00%


In [509]:
[r for r in answerable_retrieval if not r["retrieval_success"]]

[]

In [510]:
unanswerable_generation = [
    g for g in generation_results
    if not g["answerable"]
]

len(unanswerable_generation)

5

In [511]:
correct_refusals = sum(
    "don't know" in g["answer"].lower()
    for g in unanswerable_generation
)

print(correct_refusals)

5


In [512]:
refusal_accuracy = (
    correct_refusals / len(unanswerable_generation)
) * 100

print(f"Refusal Accuracy: {refusal_accuracy:.2f}%")

Refusal Accuracy: 100.00%


In [513]:
grounded_count = sum(
    g["verdict"] == "GROUNDED"
    for g in generation_results
)

groundedness_rate = (grounded_count / len(generation_results)) * 100

print(f"Groundedness Rate: {groundedness_rate:.2f}%")

Groundedness Rate: 100.00%


# Evaluation Phase

## Overview

After building the retrieval and generation pipeline, I created a small evaluation set to measure how reliably the system retrieves relevant chunks, handles questions outside the available knowledge, and generates answers grounded in the retrieved context.

The evaluation set contains **20 questions**:

* **15 answerable questions** — questions whose answers are covered by EU AI Act Article 5.
* **5 unanswerable questions** — questions whose answers are not contained in the available context.

The evaluation focuses on three metrics:

1. **Retrieval Accuracy**
2. **Refusal Accuracy**
3. **Groundedness Rate**

---

## 1. Retrieval Accuracy

### Goal

The retrieval evaluation checks whether the system retrieves the expected chunk for each answerable question.

For every question, an expected chunk ID was defined based on the content of Article 5. The `retrieve()` function then retrieves the top 3 chunks from the ChromaDB collection.

A retrieval is considered successful when the expected chunk appears among the retrieved chunk IDs.

### Initial Result

Before fixing the chunking problem:

**Retrieval Accuracy: 86.67% (13/15)**

During evaluation, I discovered that some Article 5 subsections had been incorrectly combined during preprocessing. This affected the chunk structure and caused some expected information to be stored in incorrect chunks.

### Correction

I fixed the chunking logic and rebuilt the vector database using the corrected chunks.

The chunk structure changed from **13 original chunks to 9 corrected chunks**.

The evaluation questions and their expected chunk IDs were also updated to match the corrected chunk structure.

### Final Result

**Retrieval Accuracy: 100% (15/15)**

This confirmed that all 15 answerable questions successfully retrieved their expected chunk within the top 3 results.

---

## 2. Refusal Accuracy

### Goal

A RAG system should not invent an answer when the required information is not available in its context.

For the 5 unanswerable questions, the expected behavior was for the system to explicitly state that it did not have enough information to answer.

The evaluation checked whether the generated answer contained the expected refusal phrase:

```text
"I don't know"
```

### Result

**Refusal Accuracy: 100% (5/5)**

All 5 unanswerable questions were correctly refused.

This indicates that, for this evaluation set, the generation component did not attempt to fabricate answers when the required information was outside the available context.

---

## 3. Groundedness

### Goal

Groundedness evaluates whether the claims made by the generated answer are supported by the retrieved context.

The answer and its retrieved context are passed to a separate verification step using an LLM-based verifier.

The verifier returns:

```text
GROUNDED
```

or

```text
NOT_GROUNDED
```

followed by a short explanation.

### Problem Discovered During Evaluation

The first version of the verifier incorrectly marked correct refusal answers as `NOT_GROUNDED`.

For example:

> "I don't know. The provided context does not contain information about the capital of Germany."

The answer is a refusal, and the statement that the context does not contain information about Germany's capital is consistent with the provided context.

However, the original verifier prompt did not explicitly define how refusal answers should be evaluated.

### Correction

I updated the verifier prompt with the following rule:

```text
If the answer correctly states that the context does not contain enough information
to answer the question, treat this as GROUNDED, since accurately reporting an
absence of information is itself a supported claim.
```

This made the verifier distinguish between:

* an unsupported answer to a question, and
* a correct statement that the provided context does not contain the requested information.

### Parsing Issue

During repeated evaluation, another issue was discovered in the verifier output.

The LLM sometimes returned:

```text
GROUNDED. THE ANSWER ACCURATELY REPORTS AN ABSENCE OF INFORMATION...
```

on a single line instead of placing the verdict and explanation on separate lines.

The original parser assumed that the first non-empty line contained only the verdict. As a result, the entire line could incorrectly become the stored verdict.

I therefore changed the parser to extract the canonical verdict from the beginning of the first line and keep the remaining text as the explanation.

The parser now normalizes the result to exactly:

```text
GROUNDED
```

or

```text
NOT_GROUNDED
```

This makes the evaluation robust to minor formatting variations in the LLM output.

---

## Final Evaluation Results

The complete evaluation was executed twice after the corrections.

Both runs produced the same results:

| Metric                     |           Result |
| -------------------------- | ---------------: |
| Retrieval Accuracy         | **100% (15/15)** |
| Refusal Accuracy           |   **100% (5/5)** |
| Groundedness Rate          |         **100%** |
| Total Evaluation Questions |           **20** |
| Answerable Questions       |           **15** |
| Unanswerable Questions     |            **5** |

The repeated runs are useful because LLM-based components can sometimes produce slightly different outputs between executions. In this evaluation, the same results were obtained in both runs.

---

## What This Evaluation Taught Me

This evaluation phase was not only about calculating metrics. It also helped identify weaknesses in the pipeline.

### 1. Evaluation can reveal preprocessing problems

The retrieval failures initially appeared to be retrieval problems, but investigation showed that the underlying issue was incorrect chunking.

This demonstrated that retrieval quality depends heavily on the quality of the preprocessing and chunk structure.

### 2. Evaluation criteria must define expected behavior clearly

The groundedness verifier initially did not know how to handle a correct refusal.

This showed that an evaluation system can produce misleading results if the evaluation criteria are not clearly defined.

### 3. LLM output should not be assumed to follow formatting instructions perfectly

Even when the prompt requested the verdict on the first line and the explanation on the second line, the model sometimes returned both on one line.

The parser therefore needed to be tolerant of small formatting variations.

### 4. Repeated evaluation is useful

Because parts of the system rely on an LLM, running the evaluation more than once helps identify inconsistent behavior.

In this case, two consecutive full evaluations produced identical results.

---

## Conclusion

The current evaluation set shows that the system successfully:

* retrieves the expected chunk for all 15 answerable questions,
* refuses all 5 unanswerable questions,
* and produces answers judged as grounded for the complete evaluation set.

The final evaluation results are:

**Retrieval Accuracy: 100%**

**Refusal Accuracy: 100%**

**Groundedness Rate: 100%**

These results apply specifically to the current **20-question evaluation set** and should not be interpreted as proof of production-level reliability. A larger and more diverse evaluation set would be required for stronger conclusions.

The evaluation phase also resulted in concrete improvements to the system, particularly in **chunking, evaluation design, verifier prompting, and LLM-output parsing**.


# Evaluation Conclusion

The evaluation phase was used to measure three important aspects of the RAG pipeline:

* retrieval of relevant evidence,
* appropriate refusal when information is unavailable,
* and grounding of generated answers in the retrieved context.

The final evaluation set contains **20 questions**:

* 15 answerable questions
* 5 unanswerable questions

The final results were:

| Metric             |           Result |
| ------------------ | ---------------: |
| Retrieval Accuracy | **100% (15/15)** |
| Refusal Accuracy   |   **100% (5/5)** |
| Groundedness Rate  |         **100%** |

The evaluation was run twice after the final corrections, and both runs produced the same results.

## Issues Discovered and Fixed

This evaluation phase also revealed two important problems in the pipeline.

### 1. Chunking Problem

The initial chunking logic incorrectly handled some nested Article 5 subsections.

This affected the chunk structure and caused retrieval failures.

After fixing the chunking logic, rebuilding the vector database, and updating the expected chunk mapping, Retrieval Accuracy improved from **86.67% to 100%**.

### 2. Verification Problem

The initial groundedness verifier incorrectly treated correct refusal answers as `NOT_GROUNDED`.

The verifier prompt was updated to recognize that correctly stating that the required information is absent from the provided context is itself a grounded statement.

A second issue was then found in parsing the LLM output: the model sometimes returned the verdict and explanation on the same line.

The parser was made more robust by extracting the canonical verdict (`GROUNDED` or `NOT_GROUNDED`) from the beginning of the response instead of assuming a fixed line format.

## Final Takeaway

The evaluation phase did more than produce performance metrics. It exposed weaknesses in preprocessing, evaluation criteria, and LLM output handling.

The final results show that the current system performs correctly on this **20-question evaluation set**.

However, these results should not be interpreted as proof of production-level reliability. A larger and more diverse evaluation set would be needed to make stronger claims about the system's general performance.
